# ENG EMG - how they are different


Extract window - feature and compare the T-test for each subject








In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import scipy, os
from sklearn.metrics import confusion_matrix
import seaborn as sns

import sys
sys.path.append('../../Share/')
sys.path.append('../../Share/Manual_processing/')
import baseline, config, Model, utils, Same_with_MATLAB, Feature_info

import warnings
warnings.filterwarnings('ignore')


def restore_labels(mat, labels_windowed):

    original_length = mat['Data_ADC'].shape[1]
    win_size = 1000 #Original
    win_step = 50
    #valid_length = original_length - 2 * 60

    label_full = np.zeros(original_length, dtype=labels_windowed.dtype) # 복원될 시계열 레이블 (원본 길이)

    # 슬라이딩 윈도우 인덱스 따라 레이블 채워넣기
    for i, label in enumerate(labels_windowed):
        start = 60 + i * win_step
        end = start + win_size
        if end <= original_length - 60:
            label_full[start:end] = label

    return label_full


def filtering_zero(X, y, erase_label):
    # 1. erase_label 제거
    keep_indices = y != erase_label
    X = X[keep_indices]
    y = y[keep_indices]

    # 2. erase_label보다 큰 값은 1씩 감소
    y = np.where(y > erase_label, y - 1, y)

    return X, y


from collections import Counter

def balance_data(X, y):
    # Count samples per class
    class_counts = Counter(y)
    min_count = min(class_counts.values())  # target: balance all to minority count

    indices_list = []

    for label in sorted(class_counts.keys()):
        label_indices = np.where(y == label)[0]
        selected_indices = np.random.choice(label_indices, size=min_count, replace=False)
        indices_list.extend(selected_indices)

    # Shuffle all selected indices
    balanced_indices = np.random.permutation(indices_list)

    # Subset the data
    X_balanced = X[balanced_indices]
    y_balanced = y[balanced_indices]

    return X_balanced, y_balanced

In [15]:
def return_X_y(path, balance):
    fs, lower_cutoff, upper_cutoff = Feature_info.fs, Feature_info.lower_cutoff, Feature_info.upper_cutoff
    # fs, lower_cutoff, upper_cutoff = Feature_info.fs, 1, 300
    filter_b, filter_a = Same_with_MATLAB.cheby2(4, 30, [lower_cutoff / (fs/2), upper_cutoff / (fs/2)], btype='bandpass')

    data_per_class_files = os.listdir(path)
    X, y = [], []

    for cls in data_per_class_files:
        input_path = path+cls+'/'
        files = os.listdir(input_path)
        mat = scipy.io.loadmat(input_path+files[0])
        label = mat['Data_Cls'].reshape(-1)  # shape: (1, 1729)

        feat_mean = np.tile(Feature_info.feat_mean_lst, (4, 1))
        feat_std = np.tile(Feature_info.feat_std_lst, (4, 1))

        mapped_label = np.where(label == 0, 0, int(cls))
        restored_label = restore_labels(mat, mapped_label)

        #print(mat['Data_ADC'].shape, mat['Data_Cls'].shape, restored_label.shape)
        extractor = Same_with_MATLAB.EMGFeatureExtractor(feat_mean, feat_std, filter_b, filter_a, Norm_bool=False, num_feature_set=14) #I tried 23, but not so good
        extractor.buffer = mat['Data_ADC']
        #1000, 50 = winsize and winstep
        features, labels = extractor.extract_features_with_labels(win_size=1000, win_step=50, feat_exclude=25, filtering=False, restored_label=restored_label)

        features = np.transpose(features, (2, 0, 1))  # shape: (1729, 4, 14)
        X.append(features)
        y.append(labels)
        #print(features.shape, labels.shape)

    X_train = np.concatenate(X, axis=0)
    y_train = np.concatenate(y, axis=0)
    #X_train = X_train[:, :, :, np.newaxis]

    #print(pd.Series(y_train).value_counts())
    #print(X_train.shape, y_train.shape)

    if balance:
        X_train, y_train = balance_data(X_train, y_train)

    X_train, y_train = filtering_zero(X_train, y_train, erase_label=0)
    # One_X_train = X_train[:, :, feature_idx, :]

    return X_train, y_train



def run(subject, modality):
    if modality == 'EMG': bluetooth_id = 'E8DD80E550BB'
    elif modality == 'ENG': bluetooth_id = 'E9AD0E7DCC2B'
    else:
        print("Modality should be EMG or ENG")
        return

    X_v1, y_v1 = return_X_y(path = base_path+f'{subject}/{modality}_v1/{bluetooth_id}/raw/', balance=True)
    X_v2, y_v2 = return_X_y(path = base_path+f'{subject}/{modality}_v2/{bluetooth_id}/raw/', balance=True)

    return X_v1, y_v1, X_v2, y_v2


In [16]:
base_path = 'C:/Users/hml76/PycharmProjects/MindForce/data/EMG_ENG/'

In [17]:
#48sec
X_emg1, y_emg1, X_emg2, y_emg2 = run(subject='Hunmin', modality='EMG')
X_eng1, y_eng1, X_eng2, y_eng2 = run(subject='Hunmin', modality='ENG')

In [18]:
X_emg1.shape, y_emg1.shape, X_emg2.shape, y_emg2.shape

((2104, 4, 14), (2104,), (1912, 4, 14), (1912,))

In [19]:
X_eng1.shape, y_eng1.shape, X_eng2.shape, y_eng2.shape

((1976, 4, 14), (1976,), (1112, 4, 14), (1112,))

In [20]:
gestures

array([0, 1, 2, 3, 4, 5, 6, 7, 8], dtype=int32)

In [21]:
from scipy.stats import ttest_rel
import numpy as np
import pandas as pd

feature_names = ["RMS", "VAR", "MNF"]
gestures = np.unique(y_emg1)

results = []

for g in gestures:
    idx_emg = np.where(y_emg1 == g)[0]
    idx_eng = np.where(y_eng1 == g)[0]

    # gesture별 샘플 선택
    Xg_emg = X_emg1[idx_emg]  # shape: (samples, channels, features)
    Xg_eng = X_eng1[idx_eng]

    for f, fname in enumerate(feature_names):
        # 채널 평균 후 비교
        data_emg = Xg_emg[:, :, f].mean(axis=1)
        data_eng = Xg_eng[:, :, f].mean(axis=1)

        # paired t-test (샘플 수 맞추기)
        min_len = min(len(data_emg), len(data_eng))
        t_stat, p_val = ttest_rel(data_emg[:min_len], data_eng[:min_len])

        # 평균 ± std
        mean_sd_emg = f"{data_emg[:min_len].mean():.3f} ± {data_emg[:min_len].std():.3f}"
        mean_sd_eng = f"{data_eng[:min_len].mean():.3f} ± {data_eng[:min_len].std():.3f}"

        results.append([fname, g, mean_sd_eng, mean_sd_emg, p_val])

# DataFrame 생성
df = pd.DataFrame(results, columns=["Feature", "Gesture", "ENG (Mean±SD)", "EMG (Mean±SD)", "p-value"])
print(df)


   Feature  Gesture  ENG (Mean±SD)  EMG (Mean±SD)       p-value
0      RMS        0  0.085 ± 0.015  0.041 ± 0.029  3.180815e-57
1      VAR        0  0.463 ± 0.082  0.481 ± 0.088  1.642826e-02
2      MNF        0  2.971 ± 1.130  2.127 ± 1.447  1.336588e-11
3      RMS        1  0.080 ± 0.016  0.051 ± 0.026  1.668259e-33
4      VAR        1  0.499 ± 0.030  0.484 ± 0.068  3.535639e-03
5      MNF        1  2.447 ± 0.254  1.980 ± 0.837  2.492885e-14
6      RMS        2  0.077 ± 0.011  0.042 ± 0.021  1.250158e-61
7      VAR        2  0.485 ± 0.070  0.487 ± 0.087  7.482981e-01
8      MNF        2  2.717 ± 0.911  2.140 ± 1.337  8.937998e-08
9      RMS        3  0.082 ± 0.015  0.052 ± 0.027  3.269903e-34
10     VAR        3  0.484 ± 0.053  0.448 ± 0.115  1.859229e-05
11     MNF        3  2.631 ± 0.547  3.168 ± 2.719  3.111896e-03
12     RMS        4  0.079 ± 0.013  0.044 ± 0.021  2.652476e-61
13     VAR        4  0.481 ± 0.070  0.457 ± 0.116  5.903497e-03
14     MNF        4  2.720 ± 0.729  3.18

For each gesture type - feature, average across subject + repetition


X_emg1